<a href="https://colab.research.google.com/github/muntrans/Algoritmos_de_Optimizacion/blob/main/Trabajo_Pr%C3%A1ctico_Algoritmos(JaumeMontanera).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Jaume Montanera  <br>
Url: https://github.com/.../03MAIR---Algoritmos-de-Optimizacion---/tree/master/TrabajoPractico<br>
Google Colab: https://colab.research.google.com/drive/xxxxxxxxxxxxxxxx <br>
Problema:
>1. Sesiones de doblaje <br>

Se precisa coordinar el doblaje de una película:     
- Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas.     
- Los actores de doblaje cobran todos la misma cantidad por cada día que deben esplazarse hasta el estudio de grabación independientemente del número de tomas que se graben.     
- No es posible grabar más de 6 tomas por día.      
- El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible.   

                                        

In [17]:
# Importación de librerias
import urllib.request
!pip install numpy
import numpy as np
!pip install pandas
import pandas as pd
import random

In [4]:
# Cargar los datos del problema
  # Descargo el Excel de la matriz en mi Github con formato CSV

url = "https://raw.githubusercontent.com/muntrans/Algoritmos_de_Optimizacion/refs/heads/main/TRABAJO_FINAL/Datos_problema_doblaje(30%20tomas%2C%2010%20actores).csv"
df = pd.read_csv(url, header=1, index_col=0)
# Elimino columnas que sobran
df = df.drop(columns=["Total"])
df = df.drop(columns=["Unnamed: 11"])
# Eliminao filas que sobran
df = df.drop(index=["TOTAL"])
df = df.drop(df.index[30])
# Renombro para tenerlo ordenado
df.index.name = "Toma"
df.columns.name = "Actor"
df = df.astype(int)
#display(df)


Podemos ver como la matriz muestra todas las combinaciones de Actores con sus respectivas Tomas.

In [5]:
# Información obtenida de la matriz:
# Añado las columnas que calculan los totales para cada instancia y columna
# Calcular totales
Tomas_totales = df.iloc[0:30, 0:10].sum()  # total de Tomas por Actor
Actores_totales = df.iloc[0:30, 0:10].sum(axis=1)  # total de Actores por Toma

# Crear copia y añadir totales
df_sumatorios = df.copy()
df_sumatorios["Actores por toma"] = Actores_totales
df_sumatorios.loc["Tomas por actor"] = Tomas_totales

# Ahora sí, limpiar y convertir
df_sumatorios = df_sumatorios.fillna(0)
df_sumatorios = df_sumatorios.astype(int)

display(df_sumatorios)

Actor,1,2,3,4,5,6,7,8,9,10,Actores por toma
Toma,,,,,,,,,,,
1,1,1,1,1,1,0,0,0,0,0,5
2,0,0,1,1,1,0,0,0,0,0,3
3,0,1,0,0,1,0,1,0,0,0,3
4,1,1,0,0,0,0,1,1,0,0,4
5,0,1,0,1,0,0,0,1,0,0,3
6,1,1,0,1,1,0,0,0,0,0,4
7,1,1,0,1,1,0,0,0,0,0,4
8,1,1,0,0,0,1,0,0,0,0,3
9,1,1,0,1,0,0,0,0,0,0,3


#Modelo
- **¿Como represento el espacio de soluciones?**   
Una forma sencilla para trabajar con este problema seria representar la solución en una lista. En esta lista cada posición representa una de las tomas y su valor el dia en que se grabará esa toma.   
Por lo tanto tendremos una lista de 30 elementos (cada elemento corresponde a una toma) donde sus elementos tendran valores dentre el 1 y el 5.  

- **¿Cual es la función objetivo?**   
A los **actores** debe pagarseles por dia que vayan al rodaje, independientemente de la cantidad de **tomas** que hagan en ese dia. Los actores suponen un **coste por cada dia que tengan asignado**, no por cada toma que realicen.   
El **objetivo** del problema es planificar las sesiones por dia, de manera que el **coste total sea el menor posible**.     
El **coste** de un dia de grabación viene dado por la **cantidad de actores** que asistan a ese **dia** de grabación. La cantidad de actores esta definido por la toma, cada toma tiene una cantidad de actores necesarios. Si un actor realiza más de una toma en el mismo dia, entonces ese actor solo computa como coste una vez para el dia en concreto.    
Entonces la **función objetivo** consiste en agrupar la **mayor cantidad de tomas posibles** para un mismo **actor** en un **mismo dia**. Siendo la función objetivo a minimizar el sumatorio de todos los actores por dia (_teniendo en cuenta que si un actor realiza multiples tomas solo se considera como coste una vez_).

  `coste_total = Σ (por cada día) actores_distintos_convocados_ese_día`

  La función objetivo no es lineal ya que el coste de un dia no depende solamente de sumar actores, sino de que combinaciones de tomas se agrupan juntas.
- **¿Como implemento las restricciones?**   
Todos los actores deben coincidir juntos en la misma escena, por lo tanto siempre que asignemos escenas a un dia asignaremos todos los actores que la formen a ese mismo dia.   
Teniendo en cuenta que no pueden darse más de 6 tomas por dia, deberemos limitar la cantidad de tomas a 6.


#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

Teniendo en cuenta que cada *toma* puede ser asignada a uno de los 5 posibles dias, la complejidad aumenta a medida que aumentan los dias.   
Visualmente:   

Dos tomas: PosibilidadesToma1 · PosibilidadesToma2 = 5·5 = 5^2     
Tres tomas: 5·5·5 = 5^3    
Entonces para 30 tomas = 5^30 = 931.322.574.615.478.515.625     
Podemos definir que la complejidad es de orden exponencial: O(a^n) = siendo _a_ el numero de dias entre los que repartir _n_ cantidad de tomas.

El espacio de soluciones será menor, ya que 5^30 no tiene en cuenta las restriciones, aún así seguira siendo un valor muy grande.

#### Fisibilidad inicial de modelos:
- **Búsqueda Aleatoria**:  

El método genera combinaciones aleatorias que, teniendo en cuenta las
restricciones, asigna alguno de los 5 días para cada toma.
La probabilidad de que el método encuentre la solución óptima es de
1/m, siendo m el total de soluciones posibles.    
Teniendo en cuenta que el espacio de soluciones es de un valor cercano
a 5^30, la probabilidad de dar con la solución óptima es de 1/(5^30),
siendo esta una probabilidad muy pequeña.  

- **Búsqueda Local**:    

El método explora todas las soluciones vecinas dentro del area alrededor de la mejor solución (la referencia), si encuentra una solución vecina mejor, entonces esta nueva pasa a ser la referencia, dando una nueva area local de soluciones alrededor de esta nueva.   
Teniendo en cuenta que este algorimo recorre todas las soluciones del area de la referencia, esto pude llegar a ser muy costoso en tiempo ya que el problema presenta muchas combinaciones posibles.   
Además la naturaleza de este tipo de algoritmos puede incurrir a la solución a estar en un minimo local, alejado del minimo absoluto o de otros minimos que representen soluciones mejores.  
Teniendo en cuenta el complejo espacio de soluciones, creo que es facil caer en minimos locales.      
     
- **Recocido Simulado**:     

Funciona de forma similar a la búsqueda local pero solo genera una solución vecina y añade una variable temperatura. Esta variable determina con que probabilidad el algoritmo acepta una solución vecina **peor** respecto a la solución de referencia.    
La **probabilidad de aceptar** una solución peor depende de dos factores:
  - La temperatura: cuanto más alta, mayor probabilidad de aceptar soluciones peores.
  - Cuánto peor es la solución vecina: si es solo ligeramente peor hay más probabilidad de aceptarla.

El valor de temperatura se va reduciendo a medida que avanzan las iteraciones, haciendo cada vez menos probable aceptar soluciones peores.    
Para el problema que estamos tratando este algoritmo aparentemente parece mejor opción para evitar caer en mínimos locales, es clave el ritmo con el que se determine la reducción del valor de la variable termperatura. Además a diferencia de los dos algoritmos anteriores, aqui trabajamos con una sola solución vecina, esto lo hace más rápido ya que no necesita evaluar todas las vecinas, solo una.

- **Colonia de hormigas**:   

Este tipo de algoritmo funciona bien en problemas de rutas como el TSP,
donde hay distancias entre nodos. Para este problema podríamos crear una
matriz toma x días (30 x 5), asignando probabilidades iguales inicialmente a todas las asignaciones (feromonas).    
A medida que los agentes construyen soluciones, las feromonas de las
asignaciones que forman parte de **buenas soluciones** aumentan, haciendo
que las siguientes hormigas tiendan a repetirlas. Además el algoritmo
debe incorporar una función de **evaporación** que reduce progresivamente todas las feromonas.   
Puede ser una adaptación válida pero no es lo más natural para el tipo
de problema que estamos tratando, ya que no hay distancias entre nodos.

- **Algoritmso evolutivos y genéticos**:

Inspirados en la evolución natural, son utilizados cuando el espacio de
soluciones es grande, como nuestro caso, y la función objetivo no presenta un comportamiento lineal.       
El método hace evolucionar una población de individuos (soluciones donde
cada toma tiene asignado un día) mediante tres operaciones principales:
  - **Selección**: las combinaciones más aptas, evaluadas por la función
  fitness (menor coste total de actores convocados), tienen más probabilidad de sobrevivir a la siguiente generación.

  - **Cruce**: dos soluciones se combinan para generar nuevos hijos. Por
  ejemplo, un hijo podría heredar la asignación de las tomas 1-15 de un
  padre y las tomas 16-30 del otro, generando una nueva combinación de
  días para cada toma.

  - **Mutación**: con cierta probabilidad se cambia aleatoriamente el día
  asignado a una toma, introduciendo diversidad.

Al trabajar con poblaciones de soluciones es difícil quedar atrapado en mínimos locales. En su defecto este tipo de algoritmos no puede garantizar la optimalidad.

#Diseño
- **¿Que técnica utilizo? ¿Por qué?**

Teniendo en cuenta la complejidad exponencial del problema (5^30 posibles
soluciones) y la naturaleza no lineal de la función objetivo, y la naturaleza no lineal de la función objetivo, ya que cambiar una toma de día puede afectar el coste de forma impredecible dependiendo de los actores compartidos entre tomas. Esto me hace decantar por el **Recocido Simulado** como técnica principal.

Matizando la argumentación del apartado anterior, en resumen descarto las otras alternativas por los siguientes motivos:

- **Búsqueda Aleatoria**: la probabilidad de encontrar una buena solución
en un espacio de 5^30 combinaciones es muy improbable.

- **Búsqueda Local**: para la complejidad del espacio de soluciones facilmente tenderá a quedar atrapada en mínimos locales al no
poder aceptar soluciones peores en ningún momento.

- **Colonia de Hormigas**: está diseñada para problemas de rutas con
distancias entre nodos. La adaptación a este problema mediante una matriz
de feromonas (tomas x días) puede ser forzada y poco eficiente.

- **Algoritmos Genéticos**: el cruce entre soluciones es complejo, la restricción de máximo 6 tomas por día requieren mecanismos adicionales para garantizar la validez de las soluciones generadas.

Entonces el **Recocido Simulado** resuelve las limitaciones de la búsqueda local al aceptar soluciones peores con una probabilidad controlada por la variable temperatura, escapando de mínimos locales. Adicionalmente, aprovechando gran parte del código base, implementaré también una **Búsqueda Aleatoria** para comparar resultados y demostrar la mejora que aporta el Recocido Simulado sobre una búsqueda no guiada.

- Funciones del algoritmo:

In [28]:
# Función que genere una solución aleatoria
def crear_solucion(df):
  """
  Esta función genera una combinación de días aleatoria para cada toma.
  Tiene en cuenta que no pueden haber más de 6 tomas por día.
  """
  dias_disponibles = [1, 2, 3, 4, 5]
  dias_cubiertos = []
  solucion = []

  for i in range(len(df.index)):
    dia = random.choice(dias_disponibles)
    while solucion.count(dia) >= 6:
      dias_disponibles.remove(dia)
      dias_cubiertos.append(dia)
      dia = random.choice(dias_disponibles)
    solucion.append(dia)
  return solucion

print(crear_solucion(df))

[3, 5, 1, 1, 3, 3, 5, 2, 1, 4, 1, 3, 3, 5, 5, 2, 4, 3, 5, 2, 2, 1, 2, 5, 4, 1, 4, 4, 2, 4]


In [75]:
# Función que calcule el coste de la solución
def coste_solucion(df, solucion):
  """
  La fucnión calcula el coste de una solución.
  El coste de una solución viene dado por la cantidad total
  de actores distintos que hay en cada día.
  """
  coste = 0 # el coste total sera igual a la cantidad todal de actores por dia

  for dia in range(1, 6):
    indices = []
    tomas_dia = 0
    actores_dia = 0
    for i in range(0, len(solucion)):
      if solucion[i] == dia:
        indices.append(i)
    tomas_dia = (df.iloc[indices, :].sum(axis=0)==1) # toma las filas [indices] con valor 1
    actores_dia = tomas_dia.sum() # suma todas las columnas
    coste += actores_dia
  return coste

In [80]:
# Función que implemente busqueda aleatoria
  # Al inciar un problema, cuando la variable de temperatura tenga un valor muy
  # elevado el algoritmo se comportara como un problema de busqueda aleatoria.

def busqueda_aleatoria(df, N):
  """
  Genera N soluciones aleatorias y devuelve la mejor encontrada.
  """
  mejor_solucion = crear_solucion(df)
  mejor_coste = coste_solucion(df, mejor_solucion)

  for n in range(N):
    sol_actual = crear_solucion(df)
    coste_actual = coste_solucion(df, sol_actual)
    if coste_actual < mejor_coste:
      mejor_solucion = sol_actual
      mejor_coste = coste_actual

  return mejor_solucion, mejor_coste
